# Efficient HAR on Kaggle: MobileNetV3 + Transformer on UCF101

1. **Settings → Accelerator → GPU T4** (or P100), and **Settings → Internet → On**.
2. **Add data** (right panel): search Kaggle Datasets for *UCF101* and add a copy that contains the `.avi` videos.
   The official split files (`classInd.txt`, `trainlist01.txt`, `testlist01.txt`) are used when present.
3. Run all cells. The 10-class run takes about 20-40 minutes on a T4, depending on how fast the dataset copy decodes.

In [ ]:
!git clone -q https://github.com/HunainDanish/efficient-har.git
%cd efficient-har
!pip install -q opencv-python-headless

## Locate the dataset
Finds the video folder and the split files anywhere under `/kaggle/input`.

In [ ]:
from pathlib import Path

INPUT = Path("/kaggle/input")
split_files = sorted(INPUT.rglob("classInd.txt"))
SPLIT_DIR = str(split_files[0].parent) if split_files else None

# the video root is the folder whose sub-folders hold the .avi files
video = next(INPUT.rglob("*.avi"), None)
assert video is not None, "No .avi files found: add a UCF101 dataset that contains the videos"
DATA_ROOT = str(video.parent.parent)
DATASET = "ucf101" if SPLIT_DIR else "folder"
print("DATA_ROOT:", DATA_ROOT)
print("SPLIT_DIR:", SPLIT_DIR, "| dataset mode:", DATASET)

COMMON = f"--dataset {DATASET} --data-root '{DATA_ROOT}' --max-classes 10"
if SPLIT_DIR:
    COMMON += f" --split-dir '{SPLIT_DIR}'"

## 1. Computational cost (no training needed)
Params, GFLOPs and CPU latency for each backbone × temporal module × frame count.

In [ ]:
!python benchmark.py --runs 5

## 2. Train: MobileNetV3-Small + Transformer (16 frames, 112 px)

In [ ]:
!python train.py $COMMON --frames 16 --img-size 112 --epochs 15 --batch-size 16 --workers 4 --out runs/ucf10_transformer

## 3. Baseline: same backbone with temporal averaging (TSN-style)

In [ ]:
!python train.py $COMMON --frames 16 --img-size 112 --epochs 15 --batch-size 16 --workers 4 --temporal avg --out runs/ucf10_avg

## 4. Evaluate: top-1/top-5, macro F1, confusion matrix

In [ ]:
!python evaluate.py --ckpt runs/ucf10_transformer/best.pt $COMMON --workers 4
!python evaluate.py --ckpt runs/ucf10_avg/best.pt $COMMON --workers 4

In [ ]:
import json
import matplotlib.pyplot as plt
from IPython.display import Image, display

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for run in ["ucf10_transformer", "ucf10_avg"]:
    h = json.load(open(f"runs/{run}/history.json"))
    ep = [r["epoch"] for r in h]
    ax[0].plot(ep, [r["train_loss"] for r in h], label=run)
    ax[1].plot(ep, [r["val_top1"] for r in h], label=run)
ax[0].set(title="Training loss", xlabel="epoch")
ax[1].set(title="Validation top-1", xlabel="epoch")
for a in ax:
    a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

display(Image("runs/ucf10_transformer/confusion_matrix.png"))

## 5. Results table
Combine accuracy with the cost numbers from step 1.

In [ ]:
import pandas as pd

cost = pd.read_csv("runs/benchmark.csv")
rows = []
for run, temporal in [("ucf10_transformer", "transformer"), ("ucf10_avg", "avg")]:
    m = json.load(open(f"runs/{run}/metrics.json"))
    c = cost[(cost.backbone == "mobilenet_v3_small") & (cost.temporal == temporal) & (cost.frames == 16)].iloc[0]
    rows.append({"model": f"MobileNetV3-S + {temporal}", "top1 %": round(100 * m["top1"], 2),
                 "top5 %": round(100 * m["top5"], 2), "macro F1": round(m["macro_f1"], 3),
                 "params (M)": c.params_m, "GFLOPs": c.gflops, "CPU ms/clip": c.cpu_ms})
pd.DataFrame(rows)